In [0]:
# Databricks notebook source
from pyspark.sql import DataFrame, functions as F
from pyspark.sql.window import Window

STORAGE = "stecomlakedev"
BRONZE_SQL = f"abfss://bronze@{STORAGE}.dfs.core.windows.net/sqldb"
SILVER = f"abfss://silver@{STORAGE}.dfs.core.windows.net"


def read_source(table: str) -> DataFrame:
    return spark.read.parquet(f"{BRONZE_SQL}/{table}/*/")


def standardise(df: DataFrame) -> DataFrame:
    for col in df.columns:
        snake = col.strip().lower().replace(" ", "_").replace("-", "_")
        if snake != col:
            df = df.withColumnRenamed(col, snake)
    for field in df.schema.fields:
        if str(field.dataType) == "StringType()":
            df = df.withColumn(
                field.name,
                F.when(F.trim(F.col(field.name)) == "", None)
                 .otherwise(F.trim(F.col(field.name))))
    return df


def write_silver(df: DataFrame, name: str, partition_by: str = None) -> None:
    # Overwrite, not append: this is what makes re-runs idempotent.
    writer = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    if partition_by:
        writer = writer.partitionBy(partition_by)
    writer.save(f"{SILVER}/{name}")
    n = spark.read.format("delta").load(f"{SILVER}/{name}").count()
    print(f"silver/{name}: {n} rows")

# COMMAND ----------
orders = (read_source("orders")
          .transform(standardise)
          .filter(F.col("order_id").isNotNull())
          .dropDuplicates(["order_id"])
          .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp"))
          .withColumn("order_approved_at", F.to_timestamp("order_approved_at"))
          .withColumn("order_delivered_carrier_date", F.to_timestamp("order_delivered_carrier_date"))
          .withColumn("order_delivered_customer_date", F.to_timestamp("order_delivered_customer_date"))
          .withColumn("order_estimated_delivery_date", F.to_timestamp("order_estimated_delivery_date"))
          .withColumn("order_status", F.lower(F.col("order_status")))
          .withColumn("order_date", F.to_date("order_purchase_timestamp"))
          .withColumn("delivery_delay_days",
                      F.datediff("order_delivered_customer_date",
                                 "order_estimated_delivery_date")))

write_silver(orders, "orders", partition_by="order_date")

# COMMAND ----------
# Grain is (order_id, order_item_id). Deduping on order_id alone would delete
# every line item after the first.
order_items = (read_source("order_items")
               .transform(standardise)
               .filter(F.col("order_id").isNotNull())
               .dropDuplicates(["order_id", "order_item_id"])
               .withColumn("order_item_id", F.col("order_item_id").cast("int"))
               .withColumn("price", F.col("price").cast("double"))
               .withColumn("freight_value", F.col("freight_value").cast("double"))
               .withColumn("shipping_limit_date", F.to_timestamp("shipping_limit_date"))
               # Null the implausible value rather than dropping the row, so the
               # order survives and the gap stays visible in aggregates.
               .withColumn("price", F.when(F.col("price") < 0, None).otherwise(F.col("price")))
               .withColumn("item_total", F.col("price") + F.col("freight_value")))

write_silver(order_items, "order_items")

# COMMAND ----------
# customer_id is per-order; customer_unique_id identifies the person and is what
# the gold SCD2 dimension keys on.
customers = (read_source("customers")
             .transform(standardise)
             .filter(F.col("customer_id").isNotNull())
             .dropDuplicates(["customer_id"])
             .withColumn("customer_city", F.initcap(F.col("customer_city")))
             .withColumn("customer_state", F.upper(F.col("customer_state")))
             .withColumn("customer_zip_code_prefix",
                         F.lpad(F.col("customer_zip_code_prefix").cast("string"), 5, "0")))

write_silver(customers, "customers")

# COMMAND ----------
products = (read_source("products")
            .transform(standardise)
            .filter(F.col("product_id").isNotNull())
            .dropDuplicates(["product_id"])
            .withColumnRenamed("product_category_name", "category_name")
            .withColumn("category_name", F.coalesce(F.col("category_name"), F.lit("unknown")))
            .withColumn("product_weight_g", F.col("product_weight_g").cast("double"))
            .withColumn("product_length_cm", F.col("product_length_cm").cast("double"))
            .withColumn("product_height_cm", F.col("product_height_cm").cast("double"))
            .withColumn("product_width_cm", F.col("product_width_cm").cast("double"))
            .withColumn("product_volume_cm3",
                        F.col("product_length_cm") * F.col("product_height_cm")
                        * F.col("product_width_cm")))

write_silver(products, "products")

# COMMAND ----------
# One order can have several payment rows, so the grain is
# (order_id, payment_sequential).
payments = (read_source("payments")
            .transform(standardise)
            .filter(F.col("order_id").isNotNull())
            .dropDuplicates(["order_id", "payment_sequential"])
            .withColumn("payment_sequential", F.col("payment_sequential").cast("int"))
            .withColumn("payment_installments", F.col("payment_installments").cast("int"))
            .withColumn("payment_value", F.col("payment_value").cast("double"))
            .withColumn("payment_type", F.lower(F.col("payment_type")))
            .filter(F.col("payment_value") >= 0))

write_silver(payments, "payments")

# COMMAND ----------
sellers = (read_source("sellers")
           .transform(standardise)
           .filter(F.col("seller_id").isNotNull())
           .dropDuplicates(["seller_id"])
           .withColumn("seller_city", F.initcap(F.col("seller_city")))
           .withColumn("seller_state", F.upper(F.col("seller_state"))))

write_silver(sellers, "sellers")

# COMMAND ----------
# review_id repeats in the raw data; keep the most recently answered record.
w = Window.partitionBy("review_id").orderBy(F.col("review_answer_timestamp").desc_nulls_last())

order_reviews = (read_source("order_reviews")
                 .transform(standardise)
                 .filter(F.col("review_id").isNotNull())
                 .withColumn("review_score", F.col("review_score").cast("int"))
                 .withColumn("review_creation_date", F.to_timestamp("review_creation_date"))
                 .withColumn("review_answer_timestamp", F.to_timestamp("review_answer_timestamp"))
                 .withColumn("_rn", F.row_number().over(w))
                 .filter(F.col("_rn") == 1).drop("_rn")
                 .filter(F.col("review_score").between(1, 5)))

write_silver(order_reviews, "order_reviews")

# COMMAND ----------
tables = ["orders", "order_items", "customers", "products",
          "payments", "sellers", "order_reviews"]

rows = [(t, read_source(t).count(),
         spark.read.format("delta").load(f"{SILVER}/{t}").count())
        for t in tables]

display(spark.createDataFrame(rows, ["table", "source_rows", "silver_rows"])
        .withColumn("removed", F.col("source_rows") - F.col("silver_rows")))

# COMMAND ----------
orders_s = spark.read.format("delta").load(f"{SILVER}/orders")
items_s = spark.read.format("delta").load(f"{SILVER}/order_items")
cust_s = spark.read.format("delta").load(f"{SILVER}/customers")

orphan_items = items_s.join(orders_s, "order_id", "left_anti").count()
orphan_orders = orders_s.join(cust_s, "customer_id", "left_anti").count()

print(f"orphan order_items: {orphan_items}")
print(f"orphan orders:      {orphan_orders}")
assert orphan_items == 0 and orphan_orders == 0, "referential integrity violated"